# Лабораторная работа №10. Использование анонимных функций


**Выполнил:** Ляшко Максим Александрович  

**Группа:** ИНБ-б-о-26-2  

**Вариант:** 6

---

## 1. Индивидуальное задание (Вариант 6)

**Задание:** Продемонстрируйте использование лямбда-выражений для решения задач фильтрации, сортировки и отображения с использованием наборов данных из практического занятия №7 (`p7_data_06.csv`).

> Если рядом с блокнотом лежит файл `p7_data_06.csv`, скрипт автоматически прочитает его; если файла нет, используется встроенный тестовый набор аналогичной структуры.

In [1]:
import csv
import io
import os

FILENAME = "p7_data_06.csv"

# Демонстрационный набор данных на случай, если файл p7_data_06.csv отсутствует в рабочей папке
FALLBACK_CSV = """ID;Название;Категория;Цена;Количество;Рейтинг
1;Ноутбук Pro 15;Электроника;85000;12;4.8
2;Беспроводная мышь;Периферия;2500;45;4.5
3;Механическая клавиатура;Периферия;6800;20;4.9
4;Монитор 27 IPS;Электроника;32000;8;4.7
5;USB-C Хаб;Аксессуары;3100;30;4.3
6;Веб-камера FullHD;Периферия;4900;15;4.4
7;SSD накопитель 1ТБ;Комплектующие;9500;25;4.9
8;Наушники студийные;Аудио;14200;10;4.6
"""

def load_dataset(filename: str) -> list[dict]:
    if os.path.exists(filename):
        for enc in ("utf-8-sig", "utf-8", "cp1251"):
            try:
                with open(filename, "r", encoding=enc) as f:
                    sample = f.read(1024)
                    f.seek(0)
                    delimiter = ";" if sample.count(";") > sample.count(",") else ","
                    return list(csv.DictReader(f, delimiter=delimiter))
            except UnicodeDecodeError:
                continue
    return list(csv.DictReader(io.StringIO(FALLBACK_CSV.strip()), delimiter=";"))

raw_data = load_dataset(FILENAME)
headers = list(raw_data[0].keys())

# Автоматическое определение текстового и числового столбцов для работы с p7_data_06.csv
def is_number(val: str) -> bool:
    try:
        float(str(val).replace(",", "."))
        return True
    except (ValueError, TypeError):
        return False

num_cols = [h for h in headers if all(is_number(row[h]) for row in raw_data if row[h] != "")]
str_cols = [h for h in headers if h not in num_cols]

label_col = str_cols[0] if str_cols else headers[0]
value_col = num_cols[-1] if num_cols else headers[-1]

print(f"Загружено записей: {len(raw_data)}")
print(f"Столбцы таблицы: {headers}")
print(f"Первая запись: {raw_data[0]}")

Загружено записей: 1000
Столбцы таблицы: ['Target_ID', 'Param1', 'Param2', 'Param3', 'Category', 'HasWeapon']
Первая запись: {'Target_ID': '18-3279071', 'Param1': '784', 'Param2': '97.3', 'Param3': '78.23', 'Category': 'Track', 'HasWeapon': 'true'}


In [2]:

mapped_records = list(map(
    lambda row: {
        "title": row[label_col],
        "value": float(str(row[value_col]).replace(",", ".")),
        "adjusted_value": round(float(str(row[value_col]).replace(",", ".")) * 1.1, 2)
    },
    raw_data
))

formatted_strings = list(map(
    lambda rec: f"{rec['title']}: исходное = {rec['value']}, скорректированное (+10%) = {rec['adjusted_value']}",
    mapped_records
))

print("Результат отображения (map + lambda):")
for line in formatted_strings:
    print(" ", line)

Результат отображения (map + lambda):
  18-3279071: исходное = 78.23, скорректированное (+10%) = 86.05
  17-3526500: исходное = 44.22, скорректированное (+10%) = 48.64
  82-8189823: исходное = 15.72, скорректированное (+10%) = 17.29
  95-5034249: исходное = 12.38, скорректированное (+10%) = 13.62
  69-5341531: исходное = 96.58, скорректированное (+10%) = 106.24
  23-1210643: исходное = 52.71, скорректированное (+10%) = 57.98
  40-5188387: исходное = 83.3, скорректированное (+10%) = 91.63
  38-5790428: исходное = 26.8, скорректированное (+10%) = 29.48
  32-9557410: исходное = 24.51, скорректированное (+10%) = 26.96
  37-2083120: исходное = 24.88, скорректированное (+10%) = 27.37
  54-2124486: исходное = 99.94, скорректированное (+10%) = 109.93
  43-4052203: исходное = 35.19, скорректированное (+10%) = 38.71
  57-3872356: исходное = 87.89, скорректированное (+10%) = 96.68
  50-7707342: исходное = 62.15, скорректированное (+10%) = 68.37
  41-0515882: исходное = 86.98, скорректированное (+

In [3]:

avg_value = sum(map(lambda r: r["value"], mapped_records)) / len(mapped_records)

filtered_records = list(filter(
    lambda rec: rec["value"] > avg_value,
    mapped_records
))

print(f"Среднее значение ({value_col}): {avg_value:.2f}")
print(f"Записи со значением выше среднего ({len(filtered_records)} шт.):")
for rec in filtered_records:
    print(f"  - {rec['title']}: {rec['value']}")

Среднее значение (Param3): 50.06
Записи со значением выше среднего (496 шт.):
  - 18-3279071: 78.23
  - 69-5341531: 96.58
  - 23-1210643: 52.71
  - 40-5188387: 83.3
  - 54-2124486: 99.94
  - 57-3872356: 87.89
  - 50-7707342: 62.15
  - 41-0515882: 86.98
  - 93-5482135: 69.08
  - 34-6924255: 88.99
  - 03-0157904: 89.29
  - 91-6710040: 96.05
  - 54-7452427: 77.07
  - 13-9620606: 84.35
  - 10-5292123: 64.01
  - 94-1977916: 57.75
  - 79-3874870: 84.68
  - 94-3492428: 83.24
  - 61-5338257: 84.39
  - 75-5400696: 68.17
  - 17-7925808: 95.62
  - 27-8472037: 75.09
  - 18-1607390: 94.9
  - 83-9428455: 97.7
  - 83-8625311: 53.07
  - 52-7388628: 93.38
  - 15-0067464: 86.2
  - 99-1368435: 93.62
  - 08-7859433: 90.35
  - 38-8982676: 89.6
  - 92-6019167: 70.34
  - 59-1173282: 79.61
  - 62-4925936: 51.24
  - 10-1767062: 92.9
  - 02-6611837: 52.21
  - 91-1304020: 80.76
  - 93-4769603: 85.59
  - 95-3018961: 61.49
  - 70-7198048: 78.97
  - 10-6292206: 50.82
  - 28-9953738: 52.38
  - 22-6920562: 51.21
  - 

In [4]:

sorted_desc = sorted(
    filtered_records,
    key=lambda rec: rec["value"],
    reverse=True
)


sorted_alpha = sorted(
    mapped_records,
    key=lambda rec: rec["title"].lower()
)

print("Сортировка отфильтрованных записей по убыванию значения:")
for idx, rec in enumerate(sorted_desc, 1):
    print(f"  {idx}. {rec['title']} — {rec['value']}")

print("\nСортировка всех записей по алфавиту (первые 5):")
for idx, rec in enumerate(sorted_alpha[:5], 1):
    print(f"  {idx}. {rec['title']} ({rec['value']})")

Сортировка отфильтрованных записей по убыванию значения:
  1. 54-2124486 — 99.94
  2. 05-8445679 — 99.94
  3. 05-7549240 — 99.89
  4. 95-5371790 — 99.71
  5. 73-3973497 — 99.7
  6. 99-9653346 — 99.63
  7. 92-6398702 — 99.53
  8. 37-4170880 — 99.29
  9. 64-4519300 — 99.21
  10. 92-8305090 — 98.94
  11. 13-2802158 — 98.52
  12. 90-0888102 — 98.49
  13. 31-5425184 — 98.43
  14. 13-9501291 — 98.41
  15. 31-3992217 — 98.39
  16. 42-9412497 — 98.36
  17. 56-5275406 — 98.35
  18. 13-9687398 — 98.34
  19. 05-8448385 — 98.26
  20. 05-9046798 — 98.21
  21. 09-1033988 — 98.12
  22. 03-6595591 — 98.05
  23. 96-1141756 — 97.96
  24. 05-6170998 — 97.91
  25. 21-9991863 — 97.89
  26. 97-1756949 — 97.75
  27. 83-9428455 — 97.7
  28. 97-1058770 — 97.61
  29. 35-0888934 — 97.54
  30. 61-1659534 — 97.51
  31. 12-8765790 — 97.51
  32. 69-6591407 — 97.15
  33. 18-8204502 — 97.15
  34. 34-1554371 — 96.89
  35. 13-5844796 — 96.88
  36. 50-0664757 — 96.75
  37. 17-5387274 — 96.74
  38. 54-2080880 — 96.69
  39

---
## 2. Ответы на контрольные вопросы

### 1. Что такое анонимная функция?
**Анонимная функция** (лямбда-функция) — это небольшая функция без имени, которая объявляется непосредственно в месте её использования с помощью ключевого слова `lambda` и состоит из одного выражения, результат вычисления которого автоматически возвращается.

---
### 2. Опишите синтаксис объявления лямбда-выражения.
Синтаксис лямбда-выражения в Python:
```python
lambda аргумент_1, аргумент_2, ... : выражение
```
- После ключевого слова `lambda` через запятую перечисляются входные параметры (может быть ноль, один или несколько).
- После двоеточия `:` записывается **ровно одно** выражение (без оператора `return`, циклов и многострочных конструкций) — его значение автоматически возвращается при вызове.

---
### 3. Поясните назначение лямбда-выражений. Приведите примеры их использования.
Лямбда-выражения используются для компактной записи простых одноразовых функций «на месте» без засорения пространства имён лишними объявлениями `def`. Чаще всего их передают как аргумент в функции высшего порядка: `map()`, `filter()`, `sorted()`, `min()`, `max()`.

In [5]:
# Отображение (возведение в квадрат)
nums = [1, 2, 3, 4, 5]
squares = list(map(lambda x: x ** 2, nums))
print("Квадраты:", squares)

# Сортировка списка кортежей по второму элементу
pairs = [("яблоко", 3), ("банан", 1), ("груша", 2)]
sorted_pairs = sorted(pairs, key=lambda item: item[1])
print("Сортировка:", sorted_pairs)

Квадраты: [1, 4, 9, 16, 25]
Сортировка: [('банан', 1), ('груша', 2), ('яблоко', 3)]


---
### 4. Приведите пример использования лямбда-выражений для фильтрации элементов списка.
Для фильтрации используется встроенная функция `filter(функция_условие, коллекция)`, где лямбда-выражение возвращает `True` или `False` для каждого элемента.

In [6]:
numbers = [12, 5, 8, 19, 3, 24, 7, 10]
even_gt_10 = list(filter(lambda x: x % 2 == 0 and x >= 10, numbers))
print("Чётные числа >= 10:", even_gt_10)

Чётные числа >= 10: [12, 24, 10]


---
### 5. Что будет напечатано в результате выполнения кода?
```python
sx = '1.1 3.3 6.6'
x = list(map(float, sx.split()))
print(x[1])
```
**Ответ:** Будет напечатано `3.3`.

**Пояснение:** `sx.split()` разбивает строку по пробелам на список `['1.1', '3.3', '6.6']`. Функция `map(float, ...)` преобразует каждый элемент в число с плавающей точкой `[1.1, 3.3, 6.6]`. Элемент с индексом `1` (второй по счёту) равен `3.3`.

In [7]:
sx = '1.1 3.3 6.6'
x = list(map(float, sx.split()))
print(x[1])

3.3


---
### 6.  Что будет напечатано в результате выполнения кода?
```python
data = [13, 1.3, 4, 'abs', 78]
filtered_data = list(filter(lambda x: type(x)==int, data))
print(filtered_data)
```
**Ответ:** Будет напечатано `[13, 4, 78]`.

**Пояснение:** Функция `filter` оставляет только те элементы списка `data`, чей тип строго равен `int` (целое число). Элемент `1.3` имеет тип `float`, а `'abs'` — тип `str`, поэтому они отбрасываются.

In [8]:
data = [13, 1.3, 4, 'abs', 78]
filtered_data = list(filter(lambda x: type(x) == int, data))
print(filtered_data)

[13, 4, 78]
